# Lecture 4 follow-along — Hello, Robot

Every code cell below is **verbatim** from the lecture. In class you typed these into `lec4_robot.py` and ran them with `python3 lec4_robot.py` — this notebook is for review and experiment. Run cells top to bottom.

*(down = deeper on every depth plot, as always.)*

## Build 1 — `step()`: one tick of submarine
Predict the printout before running (x = 0.5·0.1, d = (0.2+0.03)·0.1).

In [ ]:
import numpy as np
DT = 0.1                                  # tick length [s] -> 10 Hz, like the sensor

def step(state, cmd, dt=DT, disturbance=0.03):
    x, y, yaw, d = state                  # unpack: four numbers get names
    u, r, w = cmd                         # surge, yaw-rate, heave
    return np.array([x + u*np.cos(yaw)*dt,        # planar motion follows heading
                     y + u*np.sin(yaw)*dt,
                     yaw + r*dt,                  # Euler on heading
                     max(0.0, d + (w + disturbance)*dt)])  # heavy trim + hard surface

print(step(np.array([0., 0., 0., 0.]), [0.5, 0.0, 0.2]))

## Build 2 — an open-loop mission
Depth audit: 4 s × (0.25+0.03) = 1.12, then 20 s × 0.03 drift = 0.60 → **1.720 m**.

In [ ]:
mission = [([0.0, 0.0, 0.25], 4.0),   # dive for 4 s
           ([0.6, 0.0, 0.0 ], 8.0),   # cruise straight 8 s
           ([0.4, 0.35, 0.0], 6.0),   # turning leg 6 s
           ([0.6, 0.0, 0.0 ], 6.0)]   # straight again
state = np.array([0., 0., 0., 0.])    # born at the surface, pointed along x
traj = [state]                        # the story so far — page one already written
for cmd, dur in mission:              # visit each phase of the plan
    for _ in range(int(dur/DT)):      # seconds -> ticks: 4 s at 10 Hz = 40 ticks
        state = step(state, cmd); traj.append(state)   # physics, then one more story row
T = np.array(traj)                    # the list collected it; the array computes on it
print(f"final: x={T[-1,0]:.2f}  y={T[-1,1]:.2f}  depth={T[-1,3]:.3f}")   # the receipt

## It moves — a gullible robot

In [ ]:
import matplotlib.pyplot as plt; tt = np.arange(T.shape[0]) * DT  # a time stamp per row
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.2, 2.0), width_ratios=[1, 2.2])  # 2 panels
a1.plot(T[:,0], T[:,1], lw=2, color='#0e7c86'); a1.set_aspect('equal')  # meter = meter
a1.set_xlabel('x [m]'); a1.set_ylabel('y [m]'); a1.set_title('top view')
a2.plot(tt, T[:,3], lw=2, color='#e07b00'); a2.invert_yaxis()   # flipped: down = deeper
a2.set_xlabel('time [s]'); a2.set_ylabel('depth [m]  (down = deeper)')
fig.subplots_adjust(bottom=0.28, wspace=0.32); plt.savefig("topview.png", dpi=110); plt.show()

## Commanding zero and hoping
Predict: 1.0 + 0.03 × 30 = ?

In [ ]:
s = np.array([0., 0., 0., 1.0]); log = [1.0]   # start AT 1 m; log = the depth story
for _ in range(300):                           # 30 s of ticks, commanding w = 0
    s = step(s, [0.5, 0.0, 0.0]); log.append(s[3])   # tick the physics; record only depth
plt.plot(np.arange(len(log))*DT, log, c='#e07b00')   # what actually happened
plt.axhline(1.0, ls='--', c='k'); plt.gca().invert_yaxis()  # the promise; down = deeper
plt.xlabel('time [s]'); plt.ylabel('depth [m]  (down = deeper)')
plt.subplots_adjust(bottom=0.28); plt.show(); print(f"after 30 s: {log[-1]:.3f} m")

## Build 3 — close the loop (sense → think → act)

In [ ]:
rng = np.random.default_rng(7)             # Lecture 3's scramble machine
Kp, target = 0.8, 1.2
s = np.array([0., 0., 0., 0.])
zs = [0.0]
for _ in range(400):                       # 40 s
    z = s[3] + rng.normal(0, 0.05)         # SENSE (the lie, as always)
    w = Kp * (target - z)                  # THINK (the law)
    w = max(-0.4, min(0.4, w))             # clamp to real thruster authority
    s = step(s, [0.5, 0.0, w]); zs.append(s[3])   # ACT

## The rescue, plotted

In [ ]:
tt3 = np.arange(len(zs)) * DT                  # a time stamp per story entry
plt.plot(tt3, zs, lw=2, color='#0e7c86', label='P-controlled')   # what happened
plt.axhline(target, ls='--', c='k', lw=1)      # what we asked for
plt.gca().invert_yaxis(); plt.legend(loc='center right', fontsize=8)
plt.xlabel('time [s]'); plt.ylabel('depth [m]  (down = deeper)')
plt.subplots_adjust(bottom=0.28); plt.savefig("rescue.png", dpi=110); plt.show()

## Turn the dial — `run_kp` (shown on screen in lecture)
Same loop as Build 3, packaged so you can spin the knobs. Try your own Kp; compare the hold offset with the prediction 0.03/Kp.

In [ ]:
def run_kp(Kp, N=None, T_end=40.0):
    rr = np.random.default_rng(7)
    ss = np.array([0., 0., 0., 0.]); zz2 = [0.0]; ww2 = [0.0]; win = []
    for _ in range(int(T_end/DT)):
        z = ss[3] + rr.normal(0, 0.05)
        if N:
            win.append(z)
            if len(win) > N: win.pop(0)
            z = np.mean(win)
        w = max(-0.4, min(0.4, Kp*(1.2 - z)))
        ss = step(ss, [0.5, 0.0, w]); zz2.append(ss[3]); ww2.append(w)
    return np.array(zz2), np.array(ww2)

In [ ]:
for Kp in [0.1, 0.4, 0.8, 3.0]:          # the lecture's four dials
    zz2, _ = run_kp(Kp)
    plt.plot(np.arange(len(zz2))*DT, zz2, label=f'Kp={Kp}')
plt.axhline(1.2, ls='--', c='k', lw=1); plt.gca().invert_yaxis()  # target; down = deeper
plt.legend(loc='lower right', fontsize=8, ncol=4)
plt.xlabel('time [s]'); plt.ylabel('depth [m]  (down = deeper)'); plt.show()

## The filter strikes back — Lab 2's function in the loop

In [ ]:
def moving_average(z, N):                  # Lab 2's function, verbatim
    out, win = [], []
    for zi in z:
        win.append(zi)
        if len(win) > N: win.pop(0)
        out.append(sum(win) / len(win))
    return out

## Three loops, three filters
N=10: smoother commands, same depth. N=40: lag 2.0 s → the wobble triples. Lab 3 awaits.

In [ ]:
for N, c in zip([1, 10, 40], ['#56B4E9', '#009E73', '#D55E00']):  # three dials, one loop
    zzN, wwN = run_kp(0.8, N=None if N == 1 else N)    # N=1 means: believe the raw dots
    plt.plot(np.arange(len(zzN))*DT, zzN, lw=1.8, color=c, label=f'N={N}')
plt.axhline(1.2, ls='--', c='k', lw=1); plt.gca().invert_yaxis()  # target; down = deeper
plt.legend(loc='lower right', fontsize=7, ncol=3)
plt.xlabel('time [s]'); plt.ylabel('depth [m]  (down = deeper)')
plt.subplots_adjust(bottom=0.28); plt.show()